In [1]:
import sys
sys.path.append("../")

In [6]:
import torch
import logging
import matplotlib.pyplot as plt
from importlib import reload

from qmpsqsc.models import mpsqsc
from qmpsqsc.models import qmps
from qmpsqsc.models.data.utils import flip_sites_in_mps
from qmpsqsc.models.data import (
    LinearSVM,
    svm_hinge_loss,
    svm_accuracy,
    poly2_features,
    train_mpstates_with_svm,
    train_mpsqsc_with_svm,
    train_qmps_with_svm,
)
import qmpsqsc.models.data as mpsdata

logging.basicConfig(
    level=logging.WARNING,  # root stays at INFO (or WARNING)
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    force=True,
)

project_logger = logging.getLogger("qmpsqsc")
project_logger.setLevel(logging.DEBUG)
# project_logger.propagate = False  # optional: keep our debug output out of root

# Silence TenPy (or any other noisy lib) explicitly:
# logging.getLogger("tenpy").setLevel(logging.WARNING)

reload(mpsqsc)



<module 'qmpsqsc.models.mpsqsc' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/__init__.py'>

In [7]:
import torch.nn.functional as F

L = 30
chi = 2
d = 2
ghz = mpsqsc.build_ghz_state(L, d, chi).to(dtype=torch.complex128)
ghz = ghz.normalize()

zghz = ghz.copy()
zghz.As[0][:, 1] = -ghz.As[0][:, 1]

ghz_X_errors = [flip_sites_in_mps(ghz, [i], "X") for i in range(L)]
ghz_Y_errors = [flip_sites_in_mps(ghz, [i], "Y") for i in range(L)]
ghz_Z_errors = [flip_sites_in_mps(ghz, [i], "Z") for i in range(L)]

zghz_X_errors = [flip_sites_in_mps(zghz, [i], "X") for i in range(L)]
zghz_Y_errors = [flip_sites_in_mps(zghz, [i], "Y") for i in range(L)]
zghz_Z_errors = [flip_sites_in_mps(zghz, [i], "Z") for i in range(L)]

allup = mpsqsc.build_classical_state(L, d, [0]*L).to(dtype=torch.complex128)
alldown = mpsqsc.build_classical_state(L, d, [1]*L).to(dtype=torch.complex128)


In [8]:
ghzs_X = mpsqsc.add_mpstates(ghz_X_errors)
ghzs_Y = mpsqsc.add_mpstates(ghz_Y_errors)



zghzs_X = mpsqsc.add_mpstates(zghz_X_errors)
zghzs_Y = mpsqsc.add_mpstates(zghz_Y_errors)


In [9]:
from qmpsqsc.models.mpsqsc.compress import compress_mpstate

ghzs_X, ghzsX_fid = compress_mpstate(ghzs_X, 4, adam_steps=0, n_sweeps=1)
ghzs_Y, ghzsY_fid = compress_mpstate(ghzs_Y, 4, adam_steps=0, n_sweeps=1)

print("fidelities", ghzsX_fid, ghzsY_fid)

zghzs_X, zghzX_fid = compress_mpstate(zghzs_X, 4, adam_steps=0, n_sweeps=1)
zghzs_Y, zghzY_fid = compress_mpstate(zghzs_Y, 4, adam_steps=0, n_sweeps=1)

print("fidelities", zghzX_fid, zghzY_fid)


ghzs = mpsqsc.add_mpstates([ghz, ghzs_X])
ghzs = ghzs.normalize()

ghzs, ghzs_fid = compress_mpstate(ghzs, 6, adam_steps=0, n_sweeps=1)

print("fidelities", ghzs_fid)

zghzs = mpsqsc.add_mpstates([zghz, zghzs_X])
zghzs = zghzs.normalize()

zghzs, zghz_fid = compress_mpstate(zghzs, 6, adam_steps=0, n_sweeps=1)

print("fidelities", zghz_fid)


/Users/keisuke/miniconda3/envs/mpsqsc/lib/python3.11/site-packages/tenpy/networks/mps.py:1629: UserWarning: unit_cell_width is a new argument for MPS and similar classes. It is optional for now, but will become mandatory in a future release. The default value (unit_cell_width=len(sites)) is correct, iff the lattice is a Chain. For other lattices, it is incorrect. It is used for dipolar charges and correlation_function2.
  super().__init__(sites, bc, unit_cell_width)
/Users/keisuke/miniconda3/envs/mpsqsc/lib/python3.11/site-packages/tenpy/algorithms/mps_common.py:2259: UserWarning: VariationalCompression with min_sweeps=max_sweeps: we recommend to set tol_theta_diff=None to avoid overhead
  warnings.warn(
/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/compress.py:36: ComplexWarning: Casting complex values to real discards the imaginary part
  return res_mps, float(psi_t.overlap(psi))


fidelities (1.0000000000000002+0j) (1.0000000000000009+1.1940020128718855e-15j)
fidelities (0.9999999999999997+0j) (1+1.16754759853029e-15j)
fidelities (0.9999999999999998+0j)
fidelities (1.0000000000000009+0j)


In [10]:
# LinearSVM, hinge loss helpers, accuracy metrics, and polynomial feature
# maps now come from qmpsqsc.models.data.svm (see imports above).


In [11]:
def plot_loss_accuracy(losses, accuracies, title, loss_label="Loss", acc_label="Accuracy"):
    """Plot loss/accuracy curves for a training run using dual y-axes."""
    if len(losses) == 0 or len(accuracies) == 0:
        print(f"No metrics collected for {title}, skipping plot.")
        return

    steps = range(1, len(losses) + 1)
    fig, ax1 = plt.subplots(figsize=(8, 4))

    ax1.plot(steps, losses, color="tab:blue", label=loss_label)
    ax1.set_xlabel("Iteration")
    ax1.set_ylabel(loss_label, color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    ax2 = ax1.twinx()
    ax2.plot(steps, accuracies, color="tab:orange", label=acc_label)
    ax2.set_ylabel(acc_label, color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")
    ax2.set_ylim(0.0, 1.05)

    handles = ax1.lines + ax2.lines
    labels = [line.get_label() for line in handles]
    ax1.legend(handles, labels, loc="lower right")

    ax1.set_title(title)
    fig.tight_layout()
    plt.show()


In [12]:
svm = LinearSVM(in_dim=5, dtype=torch.float64)  # because phi(V) has dim 3
svm.train()

LinearSVM(
  (linear): Linear(in_features=5, out_features=1, bias=True)
)

In [13]:
data_generator = mpsdata.ghz.create_ghz_rho_batch_qsc(ghz, allup, alldown, 2**6, 0.5, random_flip=True)

In [14]:
ghzs.set_requires_grad(True)
zghzs.set_requires_grad(True)
optimizer = torch.optim.Adam(ghzs.As + zghzs.As, lr=0.001)
optim_svm = torch.optim.Adam(svm.parameters(), lr=0.08)

stage1_losses, stage1_accs = train_mpstates_with_svm(
    positive_state=ghzs,
    negative_state=zghzs,
    data_iterator=data_generator,
    svm=svm,
    mp_optimizer=optimizer,
    svm_optimizer=optim_svm,
    num_steps=50,
    svm_steps=10,
    hinge_c=0.5,
    feature_map=poly2_features,
)


2025-11-24 14:57:28,004 - qmpsqsc.models.data.svm - DEBUG - Starting hybrid training loop: steps=50, svm_steps=10, target_loss=None
2025-11-24 14:57:28,451 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 1/50 loss=0.781933 acc=0.828
2025-11-24 14:57:28,820 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 2/50 loss=0.617632 acc=0.797
2025-11-24 14:57:29,180 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 3/50 loss=0.487635 acc=0.781
2025-11-24 14:57:29,548 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 4/50 loss=0.332585 acc=0.828
2025-11-24 14:57:29,916 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 5/50 loss=0.261179 acc=0.859
2025-11-24 14:57:30,277 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 6/50 loss=0.412901 acc=1.000
2025-11-24 14:57:30,650 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 7/50 loss=0.264049 acc=1.000
2025-11-24 14:57:31,019 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 8/50 loss=0.201175 acc=1.000
2025-11-24 14:57:31,387 - qmpsqsc.models.data.svm - DEBUG - 

In [15]:
mps_qsc = mpsqsc.helper.build_qsc_from_mpstates(ghzs, zghzs)
mps_qsc = mps_qsc.truncate_bond_dimension(2)
mps_qsc.set_requires_grad(True)

In [18]:
optimizer = torch.optim.Adam(mps_qsc.As, lr=0.01)
optim_svm = torch.optim.Adam(svm.parameters(), lr=0.01)

stage2_losses, stage2_accs = train_mpsqsc_with_svm(
    model=mps_qsc,
    data_iterator=data_generator,
    svm=svm,
    model_optimizer=optimizer,
    svm_optimizer=optim_svm,
    num_steps=100,
    svm_steps=30,
    hinge_c=0.5,
    feature_map=poly2_features,
)


2025-11-24 14:58:06,213 - qmpsqsc.models.data.svm - DEBUG - Starting hybrid training loop: steps=100, svm_steps=30, target_loss=None
2025-11-24 14:58:06,362 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 1/100 loss=0.296164 acc=0.953
2025-11-24 14:58:06,509 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 2/100 loss=0.484743 acc=0.922
2025-11-24 14:58:06,658 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 3/100 loss=0.288158 acc=0.969
2025-11-24 14:58:06,805 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 4/100 loss=0.394986 acc=0.969
2025-11-24 14:58:06,950 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 5/100 loss=0.293491 acc=0.984
2025-11-24 14:58:07,103 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 6/100 loss=0.252696 acc=0.969
2025-11-24 14:58:07,323 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 7/100 loss=0.216901 acc=0.984
2025-11-24 14:58:07,471 - qmpsqsc.models.data.svm - DEBUG - Hybrid step 8/100 loss=0.344550 acc=0.906
2025-11-24 14:58:07,628 - qmpsqsc.models.data.svm -

In [51]:
mps_qsc = mps_qsc.canonicalize(truncate=True)
Us, last = qmps.construct_unitary_from_As(mps_qsc.As)
qmps_ghz = qmps.qMPS(L, chi, d, Us=Us, last_unitary=last)
qmps_ghz.set_requires_grad(True)

In [52]:
svm_qmps = svm.clone()

In [53]:
# states, labels, errors = next(data_generator)

# qmps_ghz.predict(states)[0][labels==0]

In [ ]:
from qmpsqsc.models.qmps.optimizer import StiefelAdam

optimizer = StiefelAdam(qmps_ghz.unitaries(), lr=0.01)
optim_svm = torch.optim.Adam(svm_qmps.parameters(), lr=0.01)

initial_weight = 0.0

stage3_losses, stage3_accs, stage3_meta = train_qmps_with_svm(
    model=qmps_ghz,
    data_iterator=data_generator,
    svm=svm_qmps,
    model_optimizer=optimizer,
    svm_optimizer=optim_svm,
    num_steps=2000,
    svm_steps=30,
    hinge_c=0.5,
    feature_map=poly2_features,
    # initial_weight=initial_weight,
    max_weight=1.0,
    max_weight_loops=10,
    weight_eval_batches=3,
    weight_search_tol=5e-4,
    weight_loss_threshold=0.2,
    weight_search_steps=18,
    target_loss=0.15,
    loss_window=5,
)

print(f"Initial w: {stage3_meta['initial_weight']}")
print(f"Final selected w: {stage3_meta['selected_weight']}")
print(f"Total steps: {stage3_meta['steps']}")
print("Weight loop summary:")
for idx, w in enumerate(stage3_meta.get("all_selected_weights", []), start=1):
    steps = stage3_meta.get("steps_per_loop", [])
    step_count = steps[idx - 1] if idx - 1 < len(steps) else 'n/a'
    print(f"  Loop {idx}: w={w:.3f}, steps={step_count}")
    eval_logs = stage3_meta.get("weight_evaluations_history", [])
    if idx - 1 < len(eval_logs):
        for ev_w, ev_loss in eval_logs[idx - 1]:
            print(f"    eval -> w={ev_w:.3f}, loss={ev_loss:.4f}")



2025-11-24 15:13:15,337 - qmpsqsc.models.data.svm - INFO - Starting qMPS hybrid training with weight target=1.000 (initial=0.000)
2025-11-24 15:13:15,338 - qmpsqsc.models.data.svm - DEBUG - Evaluating qMPS loss over 3 batches (hinge_c=0.5)
2025-11-24 15:13:16,461 - qmpsqsc.models.data.svm - DEBUG - Evaluating qMPS loss over 3 batches (hinge_c=0.5)
2025-11-24 15:13:17,335 - qmpsqsc.models.data.svm - DEBUG - Evaluating qMPS loss over 3 batches (hinge_c=0.5)
2025-11-24 15:13:18,243 - qmpsqsc.models.data.svm - DEBUG - Weight search: left=0.000000, right=0.499999, mid=0.499999, mid_loss=0.826691
2025-11-24 15:13:18,244 - qmpsqsc.models.data.svm - DEBUG - Evaluating qMPS loss over 3 batches (hinge_c=0.5)
2025-11-24 15:13:19,131 - qmpsqsc.models.data.svm - DEBUG - Weight search: left=0.000000, right=0.250000, mid=0.250000, mid_loss=0.586681
2025-11-24 15:13:19,131 - qmpsqsc.models.data.svm - DEBUG - Evaluating qMPS loss over 3 batches (hinge_c=0.5)
2025-11-24 15:13:20,049 - qmpsqsc.models.dat

In [ ]:
plot_loss_accuracy(stage1_losses, stage1_accs, "Stage 1: MPState SVM pre-training")
plot_loss_accuracy(stage2_losses, stage2_accs, "Stage 2: MPS-QSC fine-tuning")
plot_loss_accuracy(stage3_losses, stage3_accs, "Stage 3: qMPS Stiefel fine-tuning")
